In [1]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

D:\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
with open("blood_work.PNG", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

#image_b64[:200]

    

'iVBORw0KGgoAAAANSUhEUgAAA1EAAANbCAYAAACjHhXMAAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAFiUAABYlAUlSJPAAAP+lSURBVHhe7P27ixtJ28B///6UgQkETyDYQOBgBwcWDnZwsMKJxQY7bGDhYIWjcWDkwMwdGDl5zAYjHNyDgxUO'

In [11]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")
message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"} },
    {"type": "text", "text":"This is a blood work report. Extract all test results and flag any values outside the normal range"}
])

response = llm.invoke([message])
print(response.content)

**Blood Work Report Analysis**

The following test results have been extracted from the blood work report:

### Complete Blood Count (CBC)

* **Hemoglobin:** 15.1 g/dL (Normal: 13.5-17.5) - **Within Normal Range**
* **Hematocrit:** 44% (Normal: 41-53%) - **Within Normal Range**
* **WBC:** 6.8 x10^3/uL (Normal: 4.5-11.0) - **Within Normal Range**
* **Platelets:** 220 x10^3/uL (Normal: 150-400) - **Within Normal Range**

### Lipid Panel

* **Total Cholesterol:** 238 mg/dL (Normal: <200) - **Outside Normal Range (High)**
* **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **Outside Normal Range (High)**
* **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Outside Normal Range (Low)**
* **Triglycerides:** 188 mg/dL (Normal: <150) - **Outside Normal Range (High)**

### Metabolic Panel

* **Glucose (Fasting):** 92 mg/dL (Normal: 70-99) - **Within Normal Range**
* **HbA1c:** 5.3% (Normal: <5.7%) - **Within Normal Range**
* **Creatinine:** 1.8 mg/dL (Normal: 0.7-1.3) - **Outside Normal Range (High)

In [16]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """ Given a health condition, returns a diet plan. condition must be one of: normal, high_cholesterol, high_sugar """
    diet_plans = {
        "high_cholesterol":{
            "eat": ["fruits","vegetables","whole grains","lean protein"],
            "do_not_eat": ["red meat","","fried food","full-fat dairy","processed snacks"]
        },
        "high_sugar": {
            "eat": ["legumes","vegetables","whole grains","legumes","nuts"],
            "do_not_eat": ["white rice","","white sugar","junk food","sugar drinks"]
        },
        "normal":{
            "eat": ["fruits","vegetables","whole grains","lean protein"],
            "do_not_eat": ["excessive sugar","","processed food","trans fats"]
        }
    }
    return diet_plans.get(condition, diet_plans["normal"])
"""
agent = create_agent(
    llm,
    tools=get_diet_recommendation,
    system_prompt = "You are a helpful medical and nutritional assistant.",
    checkpointer = InMemorySaver
)
"""

'\nagent = create_agent(\n    llm,\n    tools=get_diet_recommendation,\n    system_prompt = "You are a helpful medical and nutritional assistant.",\n    checkpointer = InMemorySaver\n)\n'

In [19]:
SYSTEM_PROMPT = """ 
You are a helpful medical and nutritional assistant.
for the input blood work report image, extract the numbers and the normal range then categorize
the condition as one of: normal, high_cholesterol, high_sugar
then call appropriate tool to retrieve and present the diet plan.
"""
diet_agent = create_agent(
    llm,
    tools = [get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT    
)

result = diet_agent.invoke({"messages": HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"} },
        {"type": "text", "text":"Analyse this blood work report and suggest a diet plan."}    
    ])
})

print(result["messages"][-1].content)

Based on the blood work report, the patient has high cholesterol (238 mg/dL) and LDL cholesterol (162 mg/dL), which are above the normal range. The patient's HDL cholesterol (36 mg/dL) is below the normal range. The patient's glucose (92 mg/dL) and HbA1c (5.3%) levels are within the normal range.

Given these results, I would categorize the patient's condition as high_cholesterol.

The diet plan for a patient with high cholesterol typically involves reducing the intake of saturated and trans fats, cholesterol, and increasing the intake of soluble fiber, fruits, vegetables, and whole grains. Here is a suggested diet plan:

*   Eat:
    *   Fruits: apples, berries, citrus fruits
    *   Vegetables: leafy greens, broccoli, bell peppers
    *   Whole grains: brown rice, quinoa, whole-wheat bread
    *   Lean protein: chicken, fish, legumes
*   Do not eat:
    *   Red meat
    *   Fried food
    *   Full-fat dairy
    *   Processed snacks

This diet plan can help lower cholesterol levels an